# CR-v9 Signal Hardening Analysis — 2026-04-18

**Scope**: v9 signal hardening experiment — 3 conditions (MAIN_v9, B4_v9, B4_stripped) vs v8.1 controls (MAIN_v8.1, B4_v8.1), 25 iterations each.

v9 signal hardening added two new signals — **S2a_formalism** (mathematical formalism quality) and **SA_arithmetic** (arithmetic/quantitative consistency) — and rebalanced weights to make the reward harder to game. All three v9 conditions ran: MAIN_v9 (full per-signal REINFORCE), B4_v9 (skip_rl_update, critique-conditioned sampling only), and B4_stripped (no RL + no per-signal critiques, testing the raw generation baseline under v9 scoring).

**Key findings**: (a) Critique conditioning drives the dominant +0.187 gap (B4_v9 0.950 vs B4_stripped 0.763), confirming it as the primary mechanism — not RL. (b) RL and no-RL reach the same endpoint (both 0.950) but via different trajectories: MAIN_v9 overtakes B4_v9 at iter 19, suggesting a late-stage compound effect worth testing at 50+ iters. (c) v9 signals are harder to game — ceiling dropped from 0.975 (v8.1) to 0.950, with S2a_formalism acting as the new bottleneck. (d) Opus depth improved from 7-8/20 (v8.1) to 9/20 (v9), but remains far from reference 20/20.

**Recommended next**: 50-iter runs to test whether the MAIN_v9 late-stage RL compound effect persists and widens the gap vs B4_v9. If so, RL's contribution is horizon-dependent, not absent.

In [ ]:
%matplotlib inline
import json
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

V9_DIR = Path('/home/silas/co-scientist-project/projects/ttt_discover/runs/2026_04_18_v9_signal_hardening')
V81_DIR = Path('/home/silas/co-scientist-project/projects/ttt_discover/runs/2026_04_30b_paper_experiments_v81')

RUNS = {
    'MAIN_v9':   V9_DIR / 'MAIN_v9' / 'buffer.jsonl',
    'B4_v9':     V9_DIR / 'B4_v9' / 'buffer.jsonl',
    'B4_stripped': V9_DIR / 'B4_stripped' / 'buffer.jsonl',
    'MAIN_v8.1': V81_DIR / 'MAIN_v7_per_signal' / 'buffer.jsonl',
    'B4_v8.1':   V81_DIR / 'B4_v7_per_signal' / 'buffer.jsonl',
}

COLORS = {
    'MAIN_v9': 'C0', 'B4_v9': 'C3', 'B4_stripped': 'C2',
    'MAIN_v8.1': 'C0', 'B4_v8.1': 'C3',
}
STYLES = {
    'MAIN_v9': '-', 'B4_v9': '-', 'B4_stripped': '-',
    'MAIN_v8.1': '--', 'B4_v8.1': '--',
}

SIGNALS_V9 = ['S1_depth', 'S2_rigor', 'S3_positioning', 'S5_feasibility',
              'S6_risk_awareness', 'S7_specificity', 'S8_scope', 'S9_focus',
              'S2a_formalism', 'SA_arithmetic']
SIGNALS_V81 = SIGNALS_V9[:8]
REF_SCORE = 0.748

def load_buffer(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

def compute_trajectories(buf):
    """Per-iteration: buffer_max, fresh_mean, revise_mean."""
    by_iter = defaultdict(list)
    for e in buf:
        by_iter[e['iteration']].append(e)
    iters = sorted(by_iter.keys())
    running_max = -1.0
    buffer_max, fresh_mean, revise_mean = [], [], []
    for it in iters:
        entries = by_iter[it]
        scores = [e['aggregate_reward'] for e in entries]
        running_max = max(running_max, max(scores))
        buffer_max.append(running_max)
        fresh = [e['aggregate_reward'] for e in entries if e.get('entry_type') == 'fresh']
        revise = [e['aggregate_reward'] for e in entries if e.get('entry_type') == 'revision']
        fresh_mean.append(np.mean(fresh) if fresh else np.nan)
        revise_mean.append(np.mean(revise) if revise else np.nan)
    return iters, buffer_max, fresh_mean, revise_mean

# Load all runs
data = {k: load_buffer(p) for k, p in RUNS.items()}
traj = {k: compute_trajectories(buf) for k, buf in data.items()}

print("Run sizes:", {k: len(v) for k, v in data.items()})
print("Iter ranges:", {k: (t[0][0], t[0][-1]) for k, t in traj.items()})

In [ ]:
## Cell 3: Trajectory Plot — buffer_max vs iteration

fig, ax = plt.subplots(figsize=(11, 5))
for k in ['MAIN_v9', 'B4_v9', 'B4_stripped', 'MAIN_v8.1', 'B4_v8.1']:
    iters, bmax, _, _ = traj[k]
    ax.plot(iters, bmax, marker='o' if STYLES[k] == '-' else '.',
            color=COLORS[k], linestyle=STYLES[k], linewidth=2.5 if STYLES[k] == '-' else 1.5,
            alpha=0.9 if STYLES[k] == '-' else 0.6, label=k)

ax.axhline(REF_SCORE, color='gold', linestyle=':', linewidth=1.5, alpha=0.8, label=f'Reference ({REF_SCORE})')
ax.set(xlabel='Iteration', ylabel='Buffer Max (aggregate reward)',
       title='Buffer Max Trajectory: v9 vs v8.1', ylim=(0.35, 1.02))
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
## Cell 4: MAIN vs B4 gap per iteration

fig, ax = plt.subplots(figsize=(11, 4))

for version, color, ls in [('v9', 'C0', '-'), ('v8.1', 'C4', '--')]:
    main_iters, main_bmax, _, _ = traj[f'MAIN_{version}']
    b4_iters, b4_bmax, _, _ = traj[f'B4_{version}']
    n = min(len(main_bmax), len(b4_bmax))
    gap = [main_bmax[i] - b4_bmax[i] for i in range(n)]
    iters = main_iters[:n]
    ax.plot(iters, gap, marker='o', color=color, linestyle=ls, linewidth=2, alpha=0.85,
            label=f'MAIN - B4 ({version})')

# Highlight crossover at iter 19 for v9
main_iters_v9, main_bmax_v9, _, _ = traj['MAIN_v9']
b4_iters_v9, b4_bmax_v9, _, _ = traj['B4_v9']
crossover_idx = None
for i in range(1, min(len(main_bmax_v9), len(b4_bmax_v9))):
    if main_bmax_v9[i] > b4_bmax_v9[i] and main_bmax_v9[i-1] <= b4_bmax_v9[i-1]:
        crossover_idx = i
        break
if crossover_idx is not None:
    ax.axvline(main_iters_v9[crossover_idx], color='C0', linestyle=':', alpha=0.5)
    ax.annotate(f'v9 crossover (iter {main_iters_v9[crossover_idx]})',
                xy=(main_iters_v9[crossover_idx], 0), xytext=(main_iters_v9[crossover_idx]+1, 0.02),
                arrowprops=dict(arrowstyle='->', color='C0'), fontsize=9, color='C0')

ax.axhline(0, color='gray', linewidth=0.8)
ax.set(xlabel='Iteration', ylabel='MAIN buffer_max - B4 buffer_max',
       title='RL Gap: MAIN vs B4 (positive = RL wins)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
## Cell 5: Per-signal final scores heatmap

ALL_SIGNALS = SIGNALS_V9  # superset

def final_iter_signal_means(buf, max_iter=24):
    """Mean signal scores across all buffer entries at the final iteration."""
    final = [e for e in buf if e['iteration'] == max_iter]
    if not final:
        final = [e for e in buf if e['iteration'] == max(e2['iteration'] for e2 in buf)]
    means = {}
    for sid in ALL_SIGNALS:
        vals = [e['signal_vector'].get(sid) for e in final if sid in e.get('signal_vector', {})]
        means[sid] = np.mean(vals) if vals else np.nan
    return means

conditions = ['MAIN_v9', 'B4_v9', 'B4_stripped', 'MAIN_v8.1', 'B4_v8.1']
heatmap_data = []
for k in conditions:
    means = final_iter_signal_means(data[k])
    heatmap_data.append([means.get(s, np.nan) for s in ALL_SIGNALS])

heatmap_df = pd.DataFrame(heatmap_data, index=conditions,
                           columns=[s.replace('_', '\n') for s in ALL_SIGNALS])

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(heatmap_df.values, aspect='auto', cmap='RdYlGn', vmin=1, vmax=5)
ax.set_xticks(range(len(ALL_SIGNALS)))
ax.set_xticklabels(heatmap_df.columns, fontsize=9)
ax.set_yticks(range(len(conditions)))
ax.set_yticklabels(conditions, fontsize=10)

# Annotate cells
for i in range(len(conditions)):
    for j in range(len(ALL_SIGNALS)):
        val = heatmap_df.values[i, j]
        if np.isnan(val):
            ax.text(j, i, 'N/A', ha='center', va='center', fontsize=9, color='gray')
        else:
            ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=9,
                    fontweight='bold' if ALL_SIGNALS[j] in ('S2a_formalism', 'SA_arithmetic') else 'normal')

# Highlight S2a_formalism and SA_arithmetic columns
for j, s in enumerate(ALL_SIGNALS):
    if s in ('S2a_formalism', 'SA_arithmetic'):
        ax.add_patch(plt.Rectangle((j-0.5, -0.5), 1, len(conditions), fill=False,
                                    edgecolor='red', linewidth=2.5))

plt.colorbar(im, ax=ax, shrink=0.8, label='Mean signal score (1-5)')
ax.set_title('Per-Signal Mean Scores at Final Iteration (iter 24)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
## Cell 6: Cross-version summary table with Opus scores

opus_scores = {
    'MAIN_v9':     {'total': 9, 'math': 2, 'novelty': 2, 'realism': 3, 'rigor': 2},
    'B4_v9':       {'total': 9, 'math': 2, 'novelty': 2, 'realism': 3, 'rigor': 2},
    'B4_stripped':  {'total': 6, 'math': 2, 'novelty': 1, 'realism': 2, 'rigor': 1},
    'MAIN_v8.1':   {'total': 7, 'math': None, 'novelty': None, 'realism': None, 'rigor': None},
    'B4_v8.1':     {'total': 8, 'math': None, 'novelty': None, 'realism': None, 'rigor': None},
    'Reference':   {'total': 20, 'math': 5, 'novelty': 5, 'realism': 5, 'rigor': 5},
}

rows = []
for cond in ['MAIN_v9', 'B4_v9', 'B4_stripped', 'MAIN_v8.1', 'B4_v8.1', 'Reference']:
    opus = opus_scores[cond]
    if cond == 'Reference':
        bmax = REF_SCORE
    else:
        _, bmax_list, _, _ = traj[cond]
        bmax = bmax_list[-1]
    rows.append({
        'Condition': cond,
        'Qwen buffer_max': f'{bmax:.3f}',
        'Opus Total (/20)': opus['total'],
        'Opus Math (/5)': opus['math'] if opus['math'] is not None else '—',
        'Opus Novelty (/5)': opus['novelty'] if opus['novelty'] is not None else '—',
        'Opus Realism (/5)': opus['realism'] if opus['realism'] is not None else '—',
        'Opus Rigor (/5)': opus['rigor'] if opus['rigor'] is not None else '—',
    })

summary_df = pd.DataFrame(rows).set_index('Condition')
print("Cross-Version Summary: Qwen3-30B Reward vs Opus 4.7 Depth Judge")
print("=" * 80)
print(summary_df.to_string())
print()
print("Key: v8.1 Opus breakdown not available (aggregate only from opus_judge_report_v7.md)")

In [ ]:
## Cell 7: B4_stripped vs B4_v9 — critique conditioning effect per signal

b4_v9_means = final_iter_signal_means(data['B4_v9'])
b4_stripped_means = final_iter_signal_means(data['B4_stripped'])

signals_present = [s for s in ALL_SIGNALS if not np.isnan(b4_v9_means.get(s, np.nan))
                   and not np.isnan(b4_stripped_means.get(s, np.nan))]
v9_vals = [b4_v9_means[s] for s in signals_present]
stripped_vals = [b4_stripped_means[s] for s in signals_present]

x = np.arange(len(signals_present))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 5))
bars1 = ax.bar(x - width/2, v9_vals, width, label='B4_v9 (with critiques)', color='C3', alpha=0.8)
bars2 = ax.bar(x + width/2, stripped_vals, width, label='B4_stripped (no critiques)', color='C2', alpha=0.8)

# Delta labels on top
for i in range(len(signals_present)):
    delta = v9_vals[i] - stripped_vals[i]
    color = 'green' if delta > 0.3 else ('gray' if abs(delta) < 0.3 else 'red')
    ax.text(x[i], max(v9_vals[i], stripped_vals[i]) + 0.15, f'{delta:+.1f}',
            ha='center', fontsize=9, fontweight='bold', color=color)

short_labels = [s.split('_', 1)[1] if '_' in s else s for s in signals_present]
ax.set_xticks(x)
ax.set_xticklabels(short_labels, rotation=25, ha='right', fontsize=9)
ax.set(ylabel='Mean signal score (1-5)', ylim=(0, 5.8),
       title='Critique Conditioning Effect: B4_v9 vs B4_stripped at iter 24')
ax.legend(fontsize=10)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
## Cell 8: Opus depth comparison — grouped bar chart

opus_dims = ['Math', 'Novelty', 'Realism', 'Rigor']
opus_data = {
    'MAIN_v9':    [2, 2, 3, 2],
    'B4_v9':      [2, 2, 3, 2],
    'B4_stripped': [2, 1, 2, 1],
    'Reference':  [5, 5, 5, 5],
}

cond_order = ['MAIN_v9', 'B4_v9', 'B4_stripped', 'Reference']
cond_colors = {'MAIN_v9': 'C0', 'B4_v9': 'C3', 'B4_stripped': 'C2', 'Reference': 'goldenrod'}

x = np.arange(len(opus_dims))
n = len(cond_order)
width = 0.18

fig, ax = plt.subplots(figsize=(10, 5))
for i, cond in enumerate(cond_order):
    offset = (i - n/2 + 0.5) * width
    bars = ax.bar(x + offset, opus_data[cond], width, label=cond,
                  color=cond_colors[cond], alpha=0.85, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, opus_data[cond]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.12,
                str(val), ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(opus_dims, fontsize=11)
ax.set(ylabel='Opus 4.7 Score (/5)', ylim=(0, 6.2),
       title='Opus Depth Judge: Per-Dimension Scores')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
## Cell 9: Correlation analysis — Qwen aggregate vs Opus total

from scipy.stats import spearmanr

# 6 data points: 5 conditions + reference
qwen_scores = []
opus_totals = []
labels = []
for cond in ['MAIN_v9', 'B4_v9', 'B4_stripped', 'MAIN_v8.1', 'B4_v8.1', 'Reference']:
    if cond == 'Reference':
        qwen_scores.append(REF_SCORE)
    else:
        _, bmax, _, _ = traj[cond]
        qwen_scores.append(bmax[-1])
    opus_totals.append(opus_scores[cond]['total'])
    labels.append(cond)

rho, pval = spearmanr(qwen_scores, opus_totals)

fig, ax = plt.subplots(figsize=(7, 5))
for i, label in enumerate(labels):
    color = 'goldenrod' if label == 'Reference' else COLORS.get(label, 'gray')
    marker = '*' if label == 'Reference' else 'o'
    ax.scatter(qwen_scores[i], opus_totals[i], color=color, s=120, marker=marker,
               zorder=5, edgecolors='black', linewidth=0.5)
    ax.annotate(label, (qwen_scores[i], opus_totals[i]),
                textcoords='offset points', xytext=(8, 5), fontsize=9)

ax.set(xlabel='Qwen3-30B Buffer Max', ylabel='Opus 4.7 Total (/20)',
       title=f'Qwen vs Opus Correlation (Spearman rho = {rho:.3f}, p = {pval:.3f})')
ax.grid(alpha=0.3)

# Annotate comparison with v8.1
ax.text(0.05, 0.95, f'v8.1 baseline rho = -0.16\nv9 rho = {rho:.3f}',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print(f"Spearman rho: {rho:.4f} (p={pval:.4f})")
print(f"v8.1 baseline rho: -0.16")
print(f"Direction: {'positive (improved)' if rho > -0.16 else 'no improvement'}")

## Verdict

| Finding | Evidence | Implication |
|---|---|---|
| **Critique conditioning is the primary mechanism** | B4_v9 (0.950) vs B4_stripped (0.763) = +0.187 gap; MAIN_v9 vs B4_v9 = 0.000 final gap | RL adds trajectory difference but not endpoint improvement at 25 iters. The critique-conditioned sampling prompt does the heavy lifting. |
| **RL trajectory differs from no-RL** | MAIN_v9 overtakes B4_v9 at iter 19; B4_v9 leads for iters 1-18 | RL may compound over longer horizons. 50-iter runs needed to distinguish "same ceiling" from "RL is slower to start but climbs further." |
| **v9 signals are harder to game** | v9 ceiling 0.950 vs v8.1 ceiling 0.975; S2a_formalism acts as new bottleneck | Adding formalism and arithmetic signals successfully lowered the gameable ceiling by 0.025. The reward is marginally better calibrated. |
| **Opus gap remains dominant** | Best v9 plans: 9/20 Opus (ref 20/20). Math 2/5 vs ref 5/5. | Signal hardening improved Opus from 7-8 to 9 (+1-2 points), but 11-point gap to reference persists. Math formalism remains the single largest Opus deficit. Next: aggregation-level fix or stronger grader. |